# 1. API 설정

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

# 2. vector store 설정

In [3]:
from openai import OpenAI

client = OpenAI()

vector_store = client.vector_stores.create(
    name = "korean-history" # korean history를 바탕
)

print(f"Vector store 생성 완료")
print(f"ID {vector_store.id}")
print(f"Name {vector_store.name}")
print(f"Status {vector_store.status}")

Vector store 생성 완료
ID vs_6a1ed0d1ebf881918ae0ebb9c4d0fd85
Name korean-history
Status completed


# 3. PDF 업로드

In [4]:
from pypdf import PdfReader, PdfWriter

pdf_path = "수능특강_한국사.pdf"
output_path = "수능특강_한국사_6_7페이지.pdf"

reader = PdfReader(pdf_path)
writer = PdfWriter()

# PDF 페이지는 0부터 시작
# 6페이지 -> index 5
# 7페이지 -> index 6
for page_num in [5, 6]:
    writer.add_page(reader.pages[page_num])

with open(output_path, "wb") as f:
    writer.write(f)

print(f"저장 완료: {output_path}")

저장 완료: 수능특강_한국사_6_7페이지.pdf


In [5]:
with open(output_path, "rb") as f:
    file_upload = client.files.create(
        file=f,
        purpose="assistants" # "assistants": 지식 기반, "fine-tuning": 미세조정
    )

print("파일 업로드 완료!")
print(f"ID {file_upload.id}")
print(f"Filename {file_upload.filename}")
print(f"Size {file_upload.bytes} bytes")

파일 업로드 완료!
ID file-2AMFcf3pXTV9JsYzvvDBFv
Filename 수능특강_한국사_6_7페이지.pdf
Size 241685 bytes


In [6]:
# vector store에 파일 인덱싱
indexed_file = client.vector_stores.files.create_and_poll(
    vector_store_id = vector_store.id,
    file_id = file_upload.id
)

print(f"파일 인덱싱 완료")
print(f"ID {indexed_file.id}")
print(f"Status {indexed_file.status}")

파일 인덱싱 완료
ID file-2AMFcf3pXTV9JsYzvvDBFv
Status completed


In [7]:
# vector store 상태 확인
vs_status = client.vector_stores.retrieve(vector_store.id)

print(f"Vector store 상태")
print(f"총 파일 수: {vs_status.file_counts.total}")
print(f"완료된 파일: {vs_status.file_counts.completed}")
print(f"처리 중인 파일: {vs_status.file_counts.in_progress}")

Vector store 상태
총 파일 수: 1
완료된 파일: 1
처리 중인 파일: 0


# 4. 에이전트 만들기

In [8]:
from agents import Agent, FileSearchTool, Runner, trace

rag_agent = Agent(
    name = "RAG Expert", # RAG 전문가
    instructions = """You are a helpful assistant that answers questions based on the documents in the vector store.""",
    tools = [
        FileSearchTool(
            vector_store_ids = [vector_store.id],
            max_num_results = 5, # 상위 5개
            include_search_results = True
        )
    ]
)

print(f"Agent 생성 완료")
print(f"Name: {rag_agent.name}")
print(f"Tools: {[tool.name for tool in rag_agent.tools]}")

Agent 생성 완료
Name: RAG Expert
Tools: ['file_search']


In [9]:
async def ask_rag_agent(question: str): # 비동기: 여러 질문을 한 번에 처리할 수 있도록 함
    """RAG 에이전트에게 질문하기"""
    with trace("RAG Query"):
        result = await Runner.run(rag_agent, question) # await rag_agent.run(question)
        return result

# 첫 번째 질문하기
question1 = "청동기의 문화가 뭐야? 2문장으로 설명해줘."

print(f"질문: {question1}")
print("=" * 50)

result1 = await ask_rag_agent(question1) 
# await: 할 때까지 기다리겠다는 의미, if not -> 나중에 처리할 작업으로 넘김
print(f"답변: {result1}")

질문: 청동기의 문화가 뭐야? 2문장으로 설명해줘.
답변: RunResult:
- Last agent: Agent(name="RAG Expert", ...)
- Final output (str):
    청동기 문화는 청동으로 만든 도구와 무기를 사용하던 시대의 생활과 문화를 말해요. 이 시기에는 농경이 발달하고, 계급이 생기며, 마을이나 국가 같은 사회 조직이 점점 커졌어요.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


In [14]:
for attr in dir(result1):
    if not attr.startswith("_"):
        print(attr)

agent_tool_invocation
context_wrapper
final_output
final_output_as
input
input_guardrail_results
interruptions
last_agent
last_response_id
max_turns
new_items
output_guardrail_results
raw_responses
release_agents
to_input_list
to_state
tool_input_guardrail_results
tool_output_guardrail_results


In [16]:
print(result1.final_output)

청동기 문화는 청동으로 만든 도구와 무기를 사용하던 시대의 생활과 문화를 말해요. 이 시기에는 농경이 발달하고, 계급이 생기며, 마을이나 국가 같은 사회 조직이 점점 커졌어요.


In [17]:
# 두 번째 질문하기
question2 = "청동기 문화와 철기 문화를 비교해줘. 3문장 이내로 설명해줘"

print(f"질문: {question2}")
print("=" * 50)

result2 = await ask_rag_agent(question2) # await: 할 때까지 기다리겠다는 의미, if not -> 나중에 처리할 작업으로 넘김
print(f"답변: {result2}")

질문: 청동기 문화와 철기 문화를 비교해줘. 3문장 이내로 설명해줘
답변: RunResult:
- Last agent: Agent(name="RAG Expert", ...)
- Final output (str):
    청동기 문화는 청동 무기를 중심으로 지배층이 발달하고, 큰 무덤과 계급사회가 나타난 시기입니다.  
    철기 문화는 철제 농기구와 무기가 널리 보급되어 생산력이 크게 향상되고, 사회가 더 넓게 확산된 시기입니다.  
    즉, 청동기 문화는 지배층 중심의 사회 발달이 특징이고, 철기 문화는 생산력 향상과 사회 변화의 확대가 특징입니다.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


In [18]:
print(result2.final_output)

청동기 문화는 청동 무기를 중심으로 지배층이 발달하고, 큰 무덤과 계급사회가 나타난 시기입니다.  
철기 문화는 철제 농기구와 무기가 널리 보급되어 생산력이 크게 향상되고, 사회가 더 넓게 확산된 시기입니다.  
즉, 청동기 문화는 지배층 중심의 사회 발달이 특징이고, 철기 문화는 생산력 향상과 사회 변화의 확대가 특징입니다.


# 5. 멀티 에이전트 만들기

In [22]:
from agents import Agent, FileSearchTool, Runner, trace

# 1. rag_specialist
rag_specialist = Agent(
    name = "RAG Specialist",
    instructions = """You are a RAG specialist who excels at answering questions based on documents in the vector store. Always use the File Search Tool to find relevant information before answering.""",
    tools = [
        FileSearchTool(
            vector_store_ids = [vector_store.id],
            max_num_results = 3,
            include_search_results = True
        )
    ]
)

# 2. 일반
general_assistant = Agent(
    name="General Assistant",
    handoff_description="일반적인 질문이나 인사, 잡담 등을 처리하는 어시스턴트, 문서와 관련없는 일반적인 대화를 담당합니다.",
    instructions="""You are a friendly general assistant.

    Your role:
    - Handle greetings and casual conversations
    - Answer general knowledge questions
    - Provide helpful responses for non-document queries
    - Be friendly and conversational
    - Answer in Korean
    """, 
)

# 3. 요약
summarizer = Agent(
    name="Summarizer",
    handoff_description="논문의 전체 요약이나 특정 섹션 요약을 요청할 때 사용하는 요약 전문가", 
    instructions="""You are a summarization expert for the DeepSeek OCR paper.

    Your role:
    - Search the document and create comprehensive summaries
    - Provide structured summaries with key points
    - Highlight important findings and contributions
    - Use bullet points for clarity
    - Answer in Korean
    """, 

    tools = [
        FileSearchTool(
            vector_store_ids = [vector_store.id],
            max_num_results = 10, # 요약을 위해 더 많은 결과
            include_search_results = True
        )
    ]
)

In [23]:
triage_agent = Agent(
    name="Triage Agent",
    instructions="""You are a triage agent that routes user questions to the appropriate specialist.

    Routing rules:
    1. ** RAG Specialist **: Questions about the DeepSeek OCR paper conten , technical details, methodology, experiments, or results
    2. ** Summarizer **: Requests for summa|ries, overviews, or key takeaways
    From the paper
    3. ** General Assistant **: Greetings, general questions, or anything not related to the document

    Always analyze the user's intent carefully before routing.
    Do NOT answer questions directly - always hand off to the appropriate specialist.
    """,
    handoffs=[rag_specialist, summarizer, general_assistant]
)

print(f"Trigger Agent 생성 완료!")
print(f"handsoffs: {[agent.name for agent in triage_agent.handoffs]}")

Trigger Agent 생성 완료!
handsoffs: ['RAG Specialist', 'Summarizer', 'General Assistant']


In [24]:
async def ask_triage_agent(question: str):
    """트리아지 에이전트에게 질문하기"""
    with trace("Multi-Agent RAG"):
        result = await Runner.run(triage_agent, question)
        return result

In [ ]:
# 테스트 1: RAG
test1 = "갈돌과 갈판에 대해서 알려줘. 2문장 이내로 설명해줘."

print(f"질문: {test1}")
print("=" * 50)

result = await ask_triage_agent(test1)
print(f"답변: {result.final_output}")
print(f"최종 에이전트: {result.last_agent.name}")

Tool name 'transfer_to_RAG Specialist' contains invalid characters for function calling and has been transformed to 'transfer_to_rag_specialist'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_General Assistant' contains invalid characters for function calling and has been transformed to 'transfer_to_general_assistant'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_RAG Specialist' contains invalid characters for function calling and has been transformed to 'transfer_to_rag_specialist'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_General Assistant' contains invalid characters for function calling and has been transformed to 'transfer_to_general_assistant'. Please use only letters, digits, and underscores to avoid potential naming conflicts.


질문: 갈돌과 갈판에 대해서 알려줘. 2문장 이내로 설명해줘.
답변: 갈돌과 갈판은 선사 시대부터 곡식, 열매, 약재 등을 갈아서 가루나 반죽으로 만들 때 쓰던 도구예요. 갈돌은 손에 쥐어 문지르는 돌이고, 갈판은 그걸 받쳐 갈기 쉽게 만든 넓적한 돌입니다.
최종 에이전트: General Assistant


In [26]:
# 테스트 2: 일반 대화
test2 = "안녕!"

print(f"질문: {test2}")
print("=" * 50)

result = await ask_triage_agent(test2)
print(f"답변: {result.final_output}")
print(f"최종 에이전트: {result.last_agent.name}")

Tool name 'transfer_to_RAG Specialist' contains invalid characters for function calling and has been transformed to 'transfer_to_rag_specialist'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_General Assistant' contains invalid characters for function calling and has been transformed to 'transfer_to_general_assistant'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_RAG Specialist' contains invalid characters for function calling and has been transformed to 'transfer_to_rag_specialist'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_General Assistant' contains invalid characters for function calling and has been transformed to 'transfer_to_general_assistant'. Please use only letters, digits, and underscores to avoid potential naming conflicts.


질문: 안녕!
답변: 안녕하세요! 무엇을 도와드릴까요?
최종 에이전트: General Assistant


In [28]:
# 테스트 3: 요약 전문가
test3 = "삼국 시대의 회의제에 대해서 설명해줘. 5문장 이내로 설명해줘."

print(f"질문: {test3}")
print("=" * 50)

result = await ask_triage_agent(test3)
print(f"답변: {result.final_output}")
print(f"최종 에이전트: {result.last_agent.name}")

질문: 삼국 시대의 회의제에 대해서 설명해줘. 5문장 이내로 설명해줘.


Tool name 'transfer_to_RAG Specialist' contains invalid characters for function calling and has been transformed to 'transfer_to_rag_specialist'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_General Assistant' contains invalid characters for function calling and has been transformed to 'transfer_to_general_assistant'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_RAG Specialist' contains invalid characters for function calling and has been transformed to 'transfer_to_rag_specialist'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_General Assistant' contains invalid characters for function calling and has been transformed to 'transfer_to_general_assistant'. Please use only letters, digits, and underscores to avoid potential naming conflicts.


답변: 삼국 시대의 회의제는 국가의 중요한 일을 왕이 혼자 결정하기보다, 귀족과 관료들이 함께 모여 논의하던 정치 운영 방식입니다.  
고구려의 제가회의, 백제의 정사암 회의, 신라의 화백회의가 대표적입니다.  
이 회의들은 왕권을 견제하면서 귀족 세력의 의견을 반영하는 역할을 했습니다.  
특히 국가의 전쟁, 외교, 왕위 계승 같은 중대한 문제를 다루었습니다.  
즉, 삼국 시대 회의제는 귀족 중심의 합의 정치라고 볼 수 있습니다.
최종 에이전트: General Assistant
